## Downloading the data

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO
import os
from datetime import datetime, timedelta
import logging

In [2]:

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# -----------------------------------------
# Step 1. Get S&P 500 Tickers
# -----------------------------------------
def get_sp500_tickers() -> list:
    """
    Scrapes the S&P 500 constituents from Wikipedia.
    Wraps the HTML in a StringIO object to avoid deprecation warnings.
    Returns a list of ticker symbols with dots replaced by hyphens.
    """
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    resp = requests.get(url)
    df = pd.read_html(StringIO(resp.text))[0]
    tickers = df['Symbol'].str.strip().str.replace('.', '-', regex=False).tolist()
    logger.info(f"Retrieved {len(tickers)} S&P 500 tickers")
    return tickers

# -----------------------------------------
# Step 2. Get Sector ETF Tickers
# -----------------------------------------
def get_sector_etfs() -> list:
    """
    Returns a list of popular sector ETF tickers.
    """
    # Major sector ETFs from SPDR and other providers
    sector_etfs = [
        # SPDR Sector ETFs
        'XLK',  # Technology
        'XLF',  # Financials
        'XLV',  # Healthcare
        'XLE',  # Energy
        'XLI',  # Industrials
        'XLP',  # Consumer Staples
        'XLY',  # Consumer Discretionary
        'XLB',  # Materials
        'XLU',  # Utilities
        'XLRE', # Real Estate
        'XLC',  # Communication Services
    ]
    logger.info(f"Using {len(sector_etfs)} sector ETFs")
    return sector_etfs

# -----------------------------------------
# Step 3. Download Historical Prices
# -----------------------------------------
def download_price_data(tickers: list, start_date: str, end_date: str) -> tuple:
    """
    Downloads historical price data for multiple tickers using yfinance in one bulk call.
    Handles invalid tickers gracefully and logs errors.
    """
    valid_tickers = []
    failed_tickers = []

    # Split tickers into manageable chunks to avoid download errors
    chunk_size = 100
    ticker_chunks = [tickers[i:i + chunk_size] for i in range(0, len(tickers), chunk_size)]

    all_data = []

    for chunk in ticker_chunks:
        try:
            logger.info(f"Downloading data for {len(chunk)} tickers...")
            chunk_data = yf.download(chunk, start=start_date, end=end_date, progress=False)

            if isinstance(chunk_data.columns, pd.MultiIndex):
                # Use future_stack=True to avoid deprecation warnings
                chunk_data = chunk_data.stack(level=1, future_stack=True).rename_axis(['Date', 'symbol']).reset_index()
                chunk_valid_tickers = list(chunk_data['symbol'].unique())
                valid_tickers.extend(chunk_valid_tickers)

                # Find failed tickers in this chunk
                chunk_failed = [t for t in chunk if t not in chunk_valid_tickers]
                failed_tickers.extend(chunk_failed)

                all_data.append(chunk_data)
            else:
                # Handle single ticker case
                if not chunk_data.empty:
                    chunk_data['symbol'] = chunk[0]
                    valid_tickers.append(chunk[0])
                    all_data.append(chunk_data.reset_index())
                else:
                    failed_tickers.append(chunk[0])
        except Exception as e:
            logger.error(f"Error downloading chunk: {e}")
            failed_tickers.extend(chunk)

    if all_data:
        combined_data = pd.concat(all_data, ignore_index=True)
        logger.info(f"Successfully downloaded data for {len(valid_tickers)} tickers")
        logger.info(f"Failed to download data for {len(failed_tickers)} tickers")
        return combined_data, valid_tickers, failed_tickers
    else:
        logger.error("No data downloaded for any tickers")
        return pd.DataFrame(), [], failed_tickers

# -----------------------------------------
# Step 4. Save Data to Files
# -----------------------------------------
def save_data(data: pd.DataFrame, filename: str, directory: str = "data") -> None:
    """
    Saves data to a CSV file in the specified directory.
    Creates the directory if it doesn't exist.
    """
    # Create directory if it doesn't exist
    os.makedirs(directory, exist_ok=True)

    # Save to CSV
    filepath = os.path.join(directory, filename)
    data.to_csv(filepath, index=False)
    logger.info(f"Data saved to {filepath}")

# -----------------------------------------
# Main Function
# -----------------------------------------
def main():
    # Define date range (5 years of data)
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=10*365)).strftime('%Y-%m-%d')

    logger.info(f"Date range: {start_date} to {end_date}")

    # Get S&P 500 tickers
    sp500_tickers = get_sp500_tickers()

    # Download S&P 500 stock data
    logger.info("Downloading S&P 500 stock data...")
    stock_data, valid_stock_tickers, failed_stocks = download_price_data(sp500_tickers, start_date, end_date)

    # Save S&P 500 data
    if not stock_data.empty:
        save_data(stock_data, "stock_data.csv")
    else:
        logger.error("No S&P 500 data to save")

    # Get sector ETF tickers
    sector_etfs = get_sector_etfs()

    # Download sector ETF data
    logger.info("Downloading sector ETF data...")
    etf_data, valid_etf_tickers, failed_etfs = download_price_data(sector_etfs, start_date, end_date)

    # Save ETF data
    if not etf_data.empty:
        save_data(etf_data, "etf_data.csv")
    else:
        logger.error("No ETF data to save")

    logger.info("Data processing complete")

if __name__ == "__main__":
    main()

YF.download() has changed argument auto_adjust default to True


ERROR:yfinance:Could not get exchangeTimezoneName for ticker 'CTAS' reason: 'chart'
ERROR:yfinance:Could not get exchangeTimezoneName for ticker 'ETR' reason: 'chart'
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['CTAS', 'ETR']: YFTzMissingError('possibly delisted; no timezone found')


In [13]:
pl.read_csv(r"stock_data.csv")

Date,A.close,A.open,AAPL.close,AAPL.open,ABBV.close,ABBV.open,ABNB.close,ABNB.open,ABT.close,ABT.open,ACGL.close,ACGL.open,ACN.close,ACN.open,ADBE.close,ADBE.open,ADI.close,ADI.open,ADM.close,ADM.open,ADP.close,ADP.open,ADSK.close,ADSK.open,AEE.close,AEE.open,AEP.close,AEP.open,AES.close,AES.open,AFL.close,AFL.open,AIG.close,AIG.open,AIZ.close,AIZ.open,…,WDC.open,WEC.close,WEC.open,WELL.close,WELL.open,WFC.close,WFC.open,WM.close,WM.open,WMB.close,WMB.open,WMT.close,WMT.open,WRB.close,WRB.open,WST.close,WST.open,WTW.close,WTW.open,WY.close,WY.open,WYNN.close,WYNN.open,XEL.close,XEL.open,XOM.close,XOM.open,XYL.close,XYL.open,YUM.close,YUM.open,ZBH.close,ZBH.open,ZBRA.close,ZBRA.open,ZTS.close,ZTS.open
str,f64,f64,f64,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,str,str
"""2010-03-22""",21.563597,21.269431,6.763536,6.634736,null,null,null,null,18.598389,18.436123,7.94106,7.869214,32.215126,31.789943,34.950001,34.610001,21.185902,20.854762,19.675684,19.373806,27.600008,27.421465,29.5,29.030001,14.959089,15.034407,19.09547,19.27477,7.604416,7.69888,18.680153,18.555295,21.436073,21.71213,24.795433,24.535212,…,22.376779,15.364229,15.416238,23.812731,23.457548,20.435595,20.220554,23.345158,23.188479,8.550734,8.459567,13.495441,13.420223,5.599182,5.588241,19.098696,18.655684,61.875546,61.386177,9.806277,9.568536,49.149677,45.527987,12.718738,12.718738,37.942776,37.670823,null,null,20.547918,20.35402,50.652508,49.616831,30.129999,29.469999,null,null
"""2010-03-23""",21.813002,21.60197,6.872173,6.790318,null,null,null,null,18.760651,18.681245,7.951624,7.93472,32.192348,32.329014,35.220001,35.259998,21.439547,21.21409,19.668976,19.642143,27.649269,27.698521,29.709999,29.58,14.993855,14.941714,19.21874,19.084267,7.577425,7.59092,18.555298,18.447781,21.365456,21.602994,24.661606,24.750827,…,23.975122,15.40094,15.345871,23.807583,23.874502,20.885838,20.475916,23.631269,23.447342,8.672287,8.524141,13.560955,13.49787,5.62325,5.605746,19.35825,19.161356,62.482357,61.992994,9.884797,9.815002,49.014824,49.387268,12.745665,12.69181,37.931442,37.92578,null,null,20.601774,20.644863,50.878799,50.713438,29.950001,30.07,null,null
"""2010-03-24""",21.678694,21.729854,6.902567,6.850505,null,null,null,null,18.598389,18.729582,7.972756,7.955851,31.7292,32.108827,36.509998,37.119999,20.474306,21.362043,19.118887,19.581766,27.29834,27.513819,29.43,29.719999,14.773695,14.924329,19.045048,19.134698,7.435731,7.536943,18.607323,18.524083,21.294838,21.19212,24.892088,24.624431,…,24.686147,15.263266,15.407057,23.83847,23.781847,20.737999,20.724559,23.406464,23.549518,8.68748,8.611507,13.485737,13.522133,5.614499,5.618875,18.718334,19.331389,61.718937,62.306176,9.815004,9.843359,48.680908,48.706595,12.614021,12.715747,37.676472,37.716131,null,null,20.424036,20.617935,50.521961,50.643808,29.41,29.75,null,null
"""2010-03-25""",22.09437,21.8066,6.820715,6.949215,null,null,null,null,18.477552,18.73994,7.951624,7.983322,31.524208,31.964574,35.869999,36.66,20.389753,20.643391,18.957891,19.226227,27.076706,27.372222,29.49,29.790001,14.652026,14.814246,18.994621,19.123493,7.260298,7.449227,18.454723,18.732186,22.155104,21.641511,25.033352,24.95157,…,24.76254,15.174547,15.287745,23.81789,23.936284,20.872393,20.852233,23.345158,23.556334,8.622904,8.736863,13.493015,13.497868,5.640754,5.594805,18.566191,18.897329,62.404041,61.934247,9.631789,9.891341,47.781887,49.001955,12.602052,12.69181,37.563168,37.852112,null,null,20.494057,20.569462,50.025883,50.783061,29.030001,29.66,null,null
"""2010-03-26""",21.940891,22.158317,6.94861,6.889927,null,null,null,null,18.263506,18.539703,7.954795,7.978039,32.192348,31.364762,35.509998,35.849998,20.072712,20.389761,19.065218,19.085344,27.236769,27.101325,29.5,29.370001,14.849003,14.744719,19.10108,19.039445,7.3277

In [10]:
def download_and_pivot_combined_data(tickers: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Downloads price data and pivots it to create a combined dataframe with
    TICKER.open and TICKER.close columns for each ticker.

    Parameters:
    -----------
    tickers : list
        List of ticker symbols to download
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format

    Returns:
    --------
    pd.DataFrame
        A dataframe with date as index and columns for each ticker's open and close prices
    list
        List of tickers that were successfully downloaded
    list
        List of tickers that failed to download
    """
    # Download the data
    data, valid_tickers, failed_tickers = download_price_data(tickers, start_date, end_date)

    if data.empty:
        return None, valid_tickers, failed_tickers

    # Ensure date column is datetime
    date_col = 'Date' if 'Date' in data.columns else 'date'
    if not pd.api.types.is_datetime64_any_dtype(data[date_col]):
        data[date_col] = pd.to_datetime(data[date_col])

    # Create separate dataframes for open and close prices
    open_data = data[[date_col, 'symbol', 'Open']].copy()
    close_data = data[[date_col, 'symbol', 'Close']].copy()

    # Rename columns to prepare for merge
    open_data['column_name'] = open_data['symbol'] + '.open'
    close_data['column_name'] = close_data['symbol'] + '.close'

    # Pivot both dataframes
    open_pivot = open_data.pivot(index=date_col, columns='column_name', values='Open')
    close_pivot = close_data.pivot(index=date_col, columns='column_name', values='Close')

    # Merge the pivoted dataframes
    combined_pivot = pd.concat([open_pivot, close_pivot], axis=1)

    # Sort the columns alphabetically for easier navigation
    combined_pivot = combined_pivot.reindex(sorted(combined_pivot.columns), axis=1)

    # Sort by date
    combined_pivot = combined_pivot.sort_index()

    return combined_pivot, valid_tickers, failed_tickers

# Example usage in main function
def main():
    # Define date range
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=15*365)).strftime('%Y-%m-%d')

    # Get tickers
    sp500_tickers = get_sp500_tickers()
    sector_etfs = get_sector_etfs()

    # Download and pivot S&P 500 data with combined open/close columns
    stock_data_combined, valid_stocks, failed_stocks = download_and_pivot_combined_data(
        sp500_tickers, start_date, end_date
    )

    # Download and pivot ETF data with combined open/close columns
    etf_data_combined, valid_etfs, failed_etfs = download_and_pivot_combined_data(
        sector_etfs, start_date, end_date
    )

    # Save the combined data
    if stock_data_combined is not None:
        stock_data_combined.to_csv("stock_data.csv")
        print(f"Saved stock data with {len(valid_stocks)} tickers")

    if etf_data_combined is not None:
        etf_data_combined.to_csv("etf_data.csv")
        print(f"Saved ETF data with {len(valid_etfs)} tickers")

In [15]:
import pandas as pd
import numpy as np


# data cleaning
stock = pd.read_csv(r"stock_data.csv")
etf = pd.read_csv(r"etf_data.csv")

etf.replace("null", np.nan, inplace=True)

etf = etf.apply(pd.to_numeric, errors='ignore')

print(etf.dtypes)
print(etf.head())

Date           object
XLB.close     float64
XLB.open      float64
XLC.close     float64
XLC.open      float64
XLE.close     float64
XLE.open      float64
XLF.close     float64
XLF.open      float64
XLI.close     float64
XLI.open      float64
XLK.close     float64
XLK.open      float64
XLP.close     float64
XLP.open      float64
XLRE.close    float64
XLRE.open     float64
XLU.close     float64
XLU.open      float64
XLV.close     float64
XLV.open      float64
XLY.close     float64
XLY.open      float64
dtype: object
         Date  XLB.close   XLB.open  XLC.close  XLC.open  XLE.close  \
0  2010-03-22  24.492754  23.941539        NaN       NaN  35.019302   
1  2010-03-23  24.811880  24.514515        NaN       NaN  35.154156   
2  2010-03-24  24.739342  24.652309        NaN       NaN  34.951874   
3  2010-03-25  24.246166  24.964194        NaN       NaN  34.363422   
4  2010-03-26  24.427483  24.354956        NaN       NaN  34.375675   

    XLE.open  XLF.close  XLF.open  XLI.close  ...  XL

<ipython-input-15-d287a434f674>:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  etf = etf.apply(pd.to_numeric, errors='ignore')


In [16]:
stock.to_csv("stock_data.csv")
etf.to_csv("etf_data.csv")

In [17]:
stock.head()

,Date,A.close,A.open,AAPL.close,AAPL.open,ABBV.close,ABBV.open,ABNB.close,ABNB.open,ABT.close,...,XYL.close,XYL.open,YUM.close,YUM.open,ZBH.close,ZBH.open,ZBRA.close,ZBRA.open,ZTS.close,ZTS.open
0,2010-03-22,21.563597,21.269431,6.763536,6.634736,NaN,NaN,NaN,NaN,18.598389,...,NaN,NaN,20.547918,20.354020,50.652508,49.616831,30.129999,29.469999,NaN,NaN
1,2010-03-23,21.813002,21.601970,6.872173,6.790318,NaN,NaN,NaN,NaN,18.760651,...,NaN,NaN,20.601774,20.644863,50.878799,50.713438,29.950001,30.070000,NaN,NaN
2,2010-03-24,21.678694,21.729854,6.902567,6.850505,NaN,NaN,NaN,NaN,18.598389,...,NaN,NaN,20.424036,20.617935,50.521961,50.643808,29.410000,29.750000,NaN,NaN
3,2010-03-25,22.094370,21.806600,6.820715,6.949215,NaN,NaN,NaN,NaN,18.477552,...,NaN,NaN,20.494057,20.569462,50.025883,50.783061,29.030001,29.660000,NaN,NaN
4,2010-03-26,21.940891,22.158317,6.948610,6.889927,NaN,NaN,NaN,NaN,18.263506,...,NaN,NaN,20.612547,20.558685,49.877941,50.217364,29.320000,29.160000,NaN,NaN


In [18]:
etf.head()

,Date,XLB.close,XLB.open,XLC.close,XLC.open,XLE.close,XLE.open,XLF.close,XLF.open,XLI.close,...,XLP.close,XLP.open,XLRE.close,XLRE.open,XLU.close,XLU.open,XLV.close,XLV.open,XLY.close,XLY.open
0,2010-03-22,24.492754,23.941539,NaN,NaN,35.019302,34.669906,9.805779,9.656547,23.287149,...,18.729361,18.561595,NaN,NaN,17.782763,17.830566,25.250298,25.125835,27.204283,26.672492
1,2010-03-23,24.811880,24.514515,NaN,NaN,35.154156,35.098988,9.886611,9.830649,23.550919,...,18.876997,18.722653,NaN,NaN,17.854464,17.824587,25.374763,25.219185,27.312302,27.279067
2,2010-03-24,24.739342,24.652309,NaN,NaN,34.951874,34.921225,9.892826,9.824427,23.385124,...,18.742781,18.870283,NaN,NaN,17.669228,17.794711,25.102503,25.328092,27.204283,27.162734
3,2010-03-25,24.246166,24.964194,NaN,NaN,34.363422,35.203199,9.936354,9.961226,23.370050,...,18.715939,18.809888,NaN,NaN,17.555691,17.699101,25.001369,25.289188,27.362158,27.403703
4,2010-03-26,24.427483,24.354956,NaN,NaN,34.375675,34.479880,9.948791,9.979880,23.430346,...,18.729361,18.756204,NaN,NaN,17.597527,17.549723,24.814680,25.063606,27.470186,27.403711


In [23]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
resp = requests.get(url)
df = pd.read_html(StringIO(resp.text))[0]

# Clean ticker symbols (replace '.' with '-')
df['Symbol'] = df['Symbol'].str.strip().str.replace('.', '-', regex=False)
df["Sector"] = df['GICS Sector'].str.strip().str.replace('.', '-', regex=False)

df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Sector
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,Industrials
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,Industrials
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,Health Care
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),Health Care
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,Information Technology


In [27]:
df.to_csv("stock_metadata.csv")

In [24]:
df["GICS Sector"].unique()

array(['Industrials', 'Health Care', 'Information Technology',
       'Utilities', 'Financials', 'Materials', 'Consumer Discretionary',
       'Real Estate', 'Communication Services', 'Consumer Staples',
       'Energy'], dtype=object)

In [25]:
sector_to_etf = {
    'Information Technology': 'XLK',
    'Financials': 'XLF',
    'Health Care': 'XLV',
    'Energy': 'XLE',
    'Industrials': 'XLI',
    'Consumer Staples': 'XLP',
    'Consumer Discretionary': 'XLY',
    'Materials': 'XLB',
    'Utilities': 'XLU',
    'Real Estate': 'XLRE',
    'Communication Services': 'XLC',
}


df['ETF'] = df["GICS Sector"].map(sector_to_etf)

In [26]:
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Sector,ETF
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,Industrials,XLI
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,Industrials,XLI
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,Health Care,XLV
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),Health Care,XLV
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,Information Technology,XLK
